# Mobile Expansion Final Audit

Kiểm toán SHA-256 và đóng gói năm mô hình mở rộng sau khi tất cả Android Lock hoàn tất.

## Input bắt buộc

Gắn đúng năm output `MOB_<MODEL>_Android_Lock`: MSCOCO CLIP→S1,
F30K T1→S1 Sequence, F30K T2→S2 Sequence, F30K T1→S1 Dense Map và
F30K T1→S1 Hybrid.

Runtime: **CPU / Accelerator None / Internet Off**.


In [1]:
import hashlib, json, os, shutil
from datetime import datetime, timezone
from pathlib import Path

MODELS=['MSCOCO_CLIPS1', 'F30K_T1S1_Dense', 'F30K_T1S1_Hybrid', 'F30K_T1S1_Seq', 'F30K_T2S2_Seq']
EXPERIMENT_ID="MOB_Expansion_Final_Audit"
INPUT_ROOT=Path(os.environ.get("KLCN_INPUT_ROOT","/kaggle/input"))
OUT=Path(os.environ.get("KLCN_OUTPUT_ROOT","/kaggle/working"))/EXPERIMENT_ID
BUNDLE=OUT/"android_expansion_bundle"; BUNDLE.mkdir(parents=True,exist_ok=True)
def sha256(path):
 h=hashlib.sha256()
 with Path(path).open("rb") as f:
  for b in iter(lambda:f.read(1024*1024),b""):h.update(b)
 return h.hexdigest()
def read(p):return json.loads(Path(p).read_text(encoding="utf-8"))
def write(p,o):Path(p).write_text(json.dumps(o,ensure_ascii=False,indent=2)+"\n",encoding="utf-8")
def one(name):
 found=sorted(INPUT_ROOT.rglob(name));assert found,(name,found)
 hashes={sha256(p) for p in found};assert len(hashes)==1,(name,found)
 return found[0]
catalog=[]
for model_id in MODELS:
 exp=f"MOB_{model_id}_Android_Lock"
 complete_path=one(exp+"_COMPLETE.json"); complete=read(complete_path)
 contract_path_candidates=[p for p in INPUT_ROOT.rglob("deployment_contract.json") if p.parent.parent.name==exp]
 assert len(contract_path_candidates)==1,(model_id,contract_path_candidates)
 contract_path=contract_path_candidates[0]; contract=read(contract_path)
 assert complete["status"]=="complete" and complete["android_assets_authorized"] is True
 assert complete["model_id"]==model_id and contract["model_id"]==model_id
 assert contract["status"]=="locked" and contract["test_set_used"] is False
 assert sha256(contract_path)==complete["deployment_contract_sha256"]
 model_out=BUNDLE/model_id; model_out.mkdir(parents=True,exist_ok=True)
 copied=[]
 for record in contract["artifacts"]:
  name=Path(record["relative_path"]).name
  candidates=[p for p in contract_path.parent.glob(name) if p.is_file() and sha256(p)==record["sha256"]]
  assert len(candidates)==1,(model_id,name,candidates)
  dst=model_out/name;shutil.copy2(candidates[0],dst)
  assert dst.stat().st_size==record["bytes"] and sha256(dst)==record["sha256"]
  copied.append({"name":name,"bytes":dst.stat().st_size,"sha256":sha256(dst)})
 dst_contract=model_out/"deployment_contract.json";shutil.copy2(contract_path,dst_contract)
 catalog.append({"model_id":model_id,"teacher":contract["teacher"],"student":contract["student"],
  "dataset":contract["dataset"],"method":contract["method"],"seed":contract["seed"],
  "precision":contract["precision"],"selected_size":contract["selected_size"],
  "metrics":contract["metrics"],"artifacts":copied,"deployment_contract_sha256":sha256(dst_contract)})
assert len(catalog)==len(MODELS) and len({x["model_id"] for x in catalog})==len(MODELS)
catalog_path=BUNDLE/"mobile_model_catalog.json"
write(catalog_path,{"schema_version":"mobile_expansion_catalog_v1","created_at_utc":datetime.now(timezone.utc).isoformat(),
 "status":"complete","models":catalog,"test_set_used":False,"next_phase_authorized":"ANDROID_APP_ASSET_INTEGRATION"})
artifacts=[]
for p in sorted(BUNDLE.rglob("*")):
 if p.is_file():artifacts.append({"relative_path":str(p.relative_to(OUT)).replace("\\","/"),"bytes":p.stat().st_size,"sha256":sha256(p)})
write(OUT/"MANIFEST.json",{"schema_version":"mobile_expansion_final_manifest_v1","status":"complete",
 "models":MODELS,"artifacts":artifacts,"test_set_used":False})
write(OUT/f"{EXPERIMENT_ID}_COMPLETE.json",{"status":"complete","models":MODELS,"artifact_count":len(artifacts),
 "catalog_sha256":sha256(catalog_path),"test_set_used":False,"next_phase_authorized":"ANDROID_APP_ASSET_INTEGRATION"})
print("Status                : COMPLETE")
print("Models                :",len(catalog))
print("Artifacts             :",len(artifacts))
print("Next                  : ANDROID_APP_ASSET_INTEGRATION")


Status                : COMPLETE
Models                : 5
Artifacts             : 21
Next                  : ANDROID_APP_ASSET_INTEGRATION
